# Validação 01 — Entrada do MVP

## Goal

Comprovar o contrato inicial do MVP: uma alegação sobre saúde é obrigatória e pode ser acompanhada por um DOI ou uma URL de artigo científico.

## Setup

O notebook importa a implementação real em `src/fatofake`, sem copiar a regra de validação. Não há chamadas externas nem necessidade de credenciais.

In [1]:
from dataclasses import asdict
from pathlib import Path
from pprint import pprint
import sys

project_root = Path.cwd()
if not (project_root / "src").exists():
    project_root = project_root.parent

sys.path.insert(0, str(project_root / "src"))

from fatofake import InputValidationError, validate_analysis_input

## Steps

Validamos as três formas aceitas nesta etapa: somente alegação, alegação com DOI e alegação com URL.

In [2]:
valid_examples = [
    validate_analysis_input("Tomar café causa câncer."),
    validate_analysis_input(
        "  Vacinas   causam autismo.  ",
        "https://doi.org/10.1000/xyz123",
    ),
    validate_analysis_input(
        "Este tratamento reduz a mortalidade.",
        "https://pubmed.ncbi.nlm.nih.gov/12345678/",
    ),
]

pprint([asdict(example) for example in valid_examples])

[{'article_reference': None,
  'claim': 'Tomar café causa câncer.',
  'reference_type': None},
 {'article_reference': '10.1000/xyz123',
  'claim': 'Vacinas causam autismo.',
  'reference_type': 'doi'},
 {'article_reference': 'https://pubmed.ncbi.nlm.nih.gov/12345678/',
  'claim': 'Este tratamento reduz a mortalidade.',
  'reference_type': 'url'}]


## Checks

Os casos abaixo comprovam a normalização das entradas válidas e a rejeição de entradas que não cumprem o contrato.

In [3]:
assert valid_examples[0].article_reference is None
assert valid_examples[1].claim == "Vacinas causam autismo."
assert valid_examples[1].article_reference == "10.1000/xyz123"
assert valid_examples[1].reference_type == "doi"
assert valid_examples[2].reference_type == "url"
print("Casos válidos: aprovados")

Casos válidos: aprovados


In [4]:
invalid_examples = [
    ("", None),
    ("curta", None),
    ("Alegação válida para teste.", "artigo sem identificador"),
]

rejected = []
for claim, reference in invalid_examples:
    try:
        validate_analysis_input(claim, reference)
    except InputValidationError as error:
        rejected.append(str(error))

assert len(rejected) == len(invalid_examples)
pprint(rejected)
print("Casos inválidos: rejeitados como esperado")

['Informe uma alegação com pelo menos 8 caracteres.',
 'Informe uma alegação com pelo menos 8 caracteres.',
 'Informe um DOI ou uma URL HTTP(S) válida para o artigo.']
Casos inválidos: rejeitados como esperado


## Next Steps

O contrato de entrada estará validado quando todas as células forem executadas sem erros. A próxima etapa do fluxo será a preparação da alegação para busca científica.